# Artemus training — resumable foundation learning and instruction adaptation

This notebook runs a bounded, reproducible training cycle. With a Kaggle GPU it uses the full pilot schedule. If the account receives a CPU-only runtime, it automatically switches to a smaller FP32 CPU pilot. Both paths train ArtemisLM from random initialization on the first cycle, create separate instruction-tuned candidates, record provenance, evaluate the checkpoints, and export an auditable run bundle.

Later cycles resume **only the foundation checkpoint**, preserve its tokenizer, and use a new deterministic streaming sample. The private persistent dataset stores only the foundation checkpoint and matching tokenizer between runs.

This is a small research pilot, not frontier-scale training. FineWeb sources require attribution and terms/privacy review before model redistribution. Checkpoint promotion requires separate held-out capability evaluation.

## 1. Setup — attached offline inputs and accelerator selection

Automated runs mount the private `artemus-training-inputs` Kaggle dataset. GitHub Actions prepares the pinned/filtered corpora, source archive, wheelhouse, manifests, and evaluation inputs before the kernel is pushed; this lets the notebook run without Kaggle runtime internet or DNS.

The notebook prefers an allocated GPU. When Kaggle supplies a CPU-only runtime, it switches to a bounded FP32 pilot instead of falsely reporting a GPU run. Both paths train ArtemisLM from random initialization on the first cycle, create separate instruction-tuned candidates, record provenance, evaluate the checkpoints, and export an auditable run bundle. Later cycles resume only the verified foundation checkpoint; they never resume from an instruction candidate.


In [ ]:
import torch
from pathlib import Path
import os, subprocess, json, shutil

diagnostics = {
    "torch_version": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
}
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    result = subprocess.run(
        [nvidia_smi, "-L"], capture_output=True, text=True, check=False, timeout=10
    )
    diagnostics["nvidia_smi_exit_code"] = result.returncode
    diagnostics["nvidia_smi_output"] = (result.stdout or result.stderr).strip()
else:
    diagnostics["nvidia_smi_output"] = "nvidia-smi executable is not present"

print("Accelerator diagnostics:")
print(json.dumps(diagnostics, indent=2))

CPU_FALLBACK = not torch.cuda.is_available()
if CPU_FALLBACK:
    CPU_THREADS = max(1, min(4, os.cpu_count() or 1))
    torch.set_num_threads(CPU_THREADS)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass
    ACCELERATOR_NAME = "CPU (bounded fallback)"
    print(
        "No CUDA device was allocated. Switching to the explicitly bounded "
        f"CPU pilot with {CPU_THREADS} threads and FP32; this is real training, "
        "but not a GPU run or a full-size cycle."
    )
else:
    CPU_THREADS = None
    ACCELERATOR_NAME = torch.cuda.get_device_name(0)
    print("GPU:", ACCELERATOR_NAME)
    print("CUDA:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    print("BF16 supported:", torch.cuda.is_bf16_supported())


## 2. Load the exact project source and prepared data bundle

Automated runs attach a private Kaggle dataset containing the checked-out source commit, SentencePiece wheel, filtered English FineWeb and FineWeb-Edu splits, English-only Aya examples, OASST1 preparation, English GSM8K/MBPP task data, and held-out evaluation inputs. The notebook verifies bundle hashes and runs offline; a direct online-clone fallback remains for manual internet-enabled sessions.

In [ ]:
import hashlib
import sys

REPO_URL = 'https://github.com/FAHTHIZ/Artemus.git'
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
BUNDLE_ROOT = Path('/kaggle/input/artemus-training-inputs')
PREPARED_INPUTS_AVAILABLE = False

if BUNDLE_ROOT.is_dir() and (BUNDLE_ROOT / 'artemus-source.zip').is_file():
    # Kaggle API-launched kernels may not have outbound internet/DNS even when
    # enable_internet is requested. Use a private attached dataset containing
    # the exact source revision, prepared inputs and a local SentencePiece wheel.
    bundle_manifest_path = BUNDLE_ROOT / 'bundle-manifest.json'
    if not bundle_manifest_path.is_file():
        raise RuntimeError('Offline training input bundle is missing bundle-manifest.json.')
    bundle_manifest = json.loads(bundle_manifest_path.read_text(encoding='utf-8'))
    for item in bundle_manifest.get('artifacts', []):
        artifact = BUNDLE_ROOT / item['path']
        if not artifact.is_file() or artifact.stat().st_size != item['size_bytes']:
            raise RuntimeError(f"Offline bundle artifact is missing or has the wrong size: {item['path']}")
        digest = hashlib.sha256(artifact.read_bytes()).hexdigest()
        if digest != item['sha256']:
            raise RuntimeError(f"Offline bundle SHA-256 mismatch: {item['path']}")

    REPO_COMMIT = (BUNDLE_ROOT / 'repository-commit.txt').read_text(encoding='utf-8').strip()
    if len(REPO_COMMIT) != 40:
        raise RuntimeError('Offline bundle repository commit is not a full Git SHA.')
    WORKDIR = WORK_ROOT / f'Artemus-training-{REPO_COMMIT[:12]}-{os.getpid()}'
    WORKDIR.mkdir(parents=True, exist_ok=False)
    shutil.unpack_archive(str(BUNDLE_ROOT / 'artemus-source.zip'), str(WORKDIR))
    if not (WORKDIR / 'pyproject.toml').is_file() or not (WORKDIR / 'src' / 'artemis').is_dir():
        raise RuntimeError('Offline source archive did not unpack to a valid Artemus project.')

    wheelhouse = BUNDLE_ROOT / 'wheelhouse'
    sentencepiece_wheels = list(wheelhouse.glob('sentencepiece-*-cp313-*.whl'))
    if not sentencepiece_wheels:
        raise RuntimeError('Offline bundle is missing a CPython 3.13 SentencePiece wheel.')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-index',
        '--find-links', str(wheelhouse), 'sentencepiece>=0.2.0',
    ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-deps',
        '--no-build-isolation', '-e', str(WORKDIR),
    ], check=True)

    ARTIFACT_ROOT = WORKDIR / 'artifacts'
    ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    prepared_root = BUNDLE_ROOT / 'prepared-artifacts'
    for source_name in ('fineweb_general', 'fineweb_edu', 'aya_sft', 'oasst1', 'capability_sft'):
        archive = prepared_root / f'{source_name}.zip'
        target = ARTIFACT_ROOT / source_name
        if not archive.is_file():
            raise RuntimeError(f'Offline prepared-data archive is missing: {archive.name}')
        shutil.unpack_archive(str(archive), str(target))
        if not (target / 'manifest.json').is_file():
            raise RuntimeError(f'Offline prepared-data archive has no manifest: {source_name}')
    PREPARED_INPUTS_AVAILABLE = True
    print('Using offline Kaggle input bundle.')
    print('Artemus commit:', REPO_COMMIT)
    print('Prepared source directories:', [
        'fineweb_general', 'fineweb_edu', 'aya_sft', 'oasst1', 'capability_sft'
    ])
else:
    # Manual Kaggle UI execution may have internet enabled. Keep the direct-source
    # path as a fallback, but the automated GitHub Actions path is fully offline.
    WORKDIR = WORK_ROOT / 'Artemus-training'
    if WORKDIR.exists():
        if not (WORKDIR / '.git').is_dir():
            raise RuntimeError(f'Workspace already exists and is not a Git clone: {WORKDIR}')
        origin = subprocess.check_output(
            ['git', 'remote', 'get-url', 'origin'], cwd=WORKDIR, text=True
        ).strip().removesuffix('.git').rstrip('/')
        expected_origin = REPO_URL.removesuffix('.git').rstrip('/')
        if origin.lower() != expected_origin.lower():
            raise RuntimeError(f'Existing workspace has a different Git origin: {origin}')
        dirty = subprocess.check_output(
            ['git', 'status', '--porcelain'], cwd=WORKDIR, text=True
        ).strip()
        if dirty:
            raise RuntimeError('Existing Artemus-training clone has local changes; preserve them and use a clean workspace.')
        subprocess.run(['git', 'pull', '--ff-only'], cwd=WORKDIR, check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR)], check=True)
    REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=WORKDIR, text=True).strip()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev,data]'], cwd=WORKDIR, check=True)
    ARTIFACT_ROOT = WORKDIR / 'artifacts'

os.chdir(WORKDIR)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print('Artemus commit:', REPO_COMMIT)


## 3. Reproducible experiment settings

Keep the first run intentionally bounded. Increase document count and training steps only after the data pipeline, memory use, validation behavior, and checkpoint recovery are verified. `sample-10BT` names the upstream subset; this notebook does **not** download 10 billion tokens.


In [ ]:
DATASET_ID = 'HuggingFaceFW/fineweb-edu'
DATASET_CONFIG = 'sample-10BT'
DATASET_SPLIT = 'train'
DATASET_REVISION = '87f09149ef4734204d70ed1d046ddc9ca3f2b8f9'
DATASET_LICENSE = 'ODC-By-1.0'
LICENSE_REVIEW_NOTE = (
    'English-only research pilot. FineWeb-Edu declares ODC-By 1.0; Common Crawl terms '
    'and residual privacy/content caveats require separate review before redistribution.'
)
GENERAL_DATASET_ID = 'HuggingFaceFW/fineweb'
GENERAL_DATASET_CONFIG = 'sample-10BT'
GENERAL_DATASET_LICENSE = 'ODC-By-1.0'
GENERAL_LICENSE_REVIEW_NOTE = (
    'English-only research pilot. FineWeb declares ODC-By 1.0; Common Crawl terms '
    'and residual privacy/content caveats require separate review before redistribution.'
)
MIN_LANGUAGE_SCORE = 0.70
VALIDATION_PERCENT = 1.0
MIN_CHARS = 300
MAX_CHARS = 40_000
VOCAB_SIZE = 4096
BASE_LR = 3e-4
FINEWEB_GENERAL_WEIGHT = 0.80
FINEWEB_EDU_WEIGHT = 0.20

if CPU_FALLBACK:
    MAX_DOCUMENTS = 3_000
    GENERAL_MAX_DOCUMENTS = 3_000
    MIX_TRAIN_DOCUMENTS = 2_000
    MIX_VALIDATION_DOCUMENTS = 100
    BLOCK_SIZE = 256
    TRAIN_STEPS = 1_000
    CHECKPOINT_INTERVAL = 100
    BATCH_SIZE = 1
    GRAD_ACCUM = 1
    PRECISION = 'fp32'
    FOUNDATION_EVAL_INTERVAL = 250
    AYA_MAX_EXAMPLES = 1_500
    AYA_SFT_STEPS = 250
    AYA_EVAL_INTERVAL = 100
    TASK_SFT_STEPS = 250
    TASK_EVAL_INTERVAL = 100
    EVAL_ITERS = 2
    CAPABILITY_MAX_NEW_TOKENS = 64
    CAPABILITY_MAX_MATH_EXAMPLES = 50
    CAPABILITY_MAX_CODE_EXAMPLES = 25
    print("Training profile: CPU pilot", {
        "foundation_steps": TRAIN_STEPS, "aya_steps": AYA_SFT_STEPS,
        "task_sft_steps": TASK_SFT_STEPS, "block_size": BLOCK_SIZE,
        "batch_size": BATCH_SIZE, "threads": CPU_THREADS, "precision": PRECISION,
    })
else:
    MAX_DOCUMENTS = 20_000
    GENERAL_MAX_DOCUMENTS = 20_000
    MIX_TRAIN_DOCUMENTS = 10_000
    MIX_VALIDATION_DOCUMENTS = 500
    BLOCK_SIZE = 512
    TRAIN_STEPS = 6_000
    CHECKPOINT_INTERVAL = 100
    BATCH_SIZE = 4
    GRAD_ACCUM = 4
    PRECISION = 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'
    FOUNDATION_EVAL_INTERVAL = 250
    AYA_MAX_EXAMPLES = 10_000
    AYA_SFT_STEPS = 1_500
    AYA_EVAL_INTERVAL = 200
    TASK_SFT_STEPS = 1_000
    TASK_EVAL_INTERVAL = 150
    EVAL_ITERS = 8
    CAPABILITY_MAX_NEW_TOKENS = 96
    CAPABILITY_MAX_MATH_EXAMPLES = 100
    CAPABILITY_MAX_CODE_EXAMPLES = 50

ARTIFACT_ROOT = WORKDIR / 'artifacts'
CHECKPOINT_ROOT = WORKDIR / 'checkpoints'
GENERAL_DATA_DIR = ARTIFACT_ROOT / 'fineweb_general'
DATA_DIR = ARTIFACT_ROOT / 'fineweb_edu'
AYA_DIR = ARTIFACT_ROOT / 'aya_sft'
MIX_DIR = ARTIFACT_ROOT / 'foundation_mixture'
TOKENIZER_PREFIX = ARTIFACT_ROOT / 'artemus-foundation-tokenizer'
CHECKPOINT = CHECKPOINT_ROOT / 'artemus-foundation-fineweb-edu.pt'
GENERAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

STATE_DATASET_SLUG = 'artemus-training-state'
STATE_ROOT = Path('/kaggle/input') / STATE_DATASET_SLUG
STATE_CHECKPOINT = STATE_ROOT / 'artemus-foundation-current.pt'
STATE_TOKENIZER = STATE_ROOT / 'tokenizer.model'
STATE_MANIFEST = STATE_ROOT / 'state.json'
HAS_FOUNDATION_STATE = STATE_CHECKPOINT.is_file()
HAS_TOKENIZER_STATE = STATE_TOKENIZER.is_file()
if HAS_FOUNDATION_STATE != HAS_TOKENIZER_STATE:
    raise RuntimeError('Training state is incomplete: checkpoint and tokenizer must be present together.')
if HAS_FOUNDATION_STATE and not STATE_MANIFEST.is_file():
    raise RuntimeError('Training state has checkpoints but no state.json integrity manifest.')
if STATE_MANIFEST.is_file() and not HAS_FOUNDATION_STATE:
    raise RuntimeError('Training state has a manifest but is missing its foundation checkpoint.')

STATE_PAYLOAD = None
if HAS_FOUNDATION_STATE:
    import base64
    import hashlib
    STATE_RECORD = json.loads(STATE_MANIFEST.read_text(encoding='utf-8'))
    STATE_PAYLOAD = torch.load(STATE_CHECKPOINT, map_location='cpu', weights_only=True)
    if STATE_PAYLOAD.get('format_version') != 3:
        raise RuntimeError('Only provenance-enabled checkpoint format v3 can be resumed.')
    if STATE_PAYLOAD.get('response_only') is True:
        raise RuntimeError('Persistent state must contain a foundation checkpoint, not an instruction-SFT checkpoint.')
    STATE_METADATA = STATE_PAYLOAD.get('run_metadata') or {}
    STATE_LINEAGE = STATE_METADATA.get('lineage') or {}
    if STATE_LINEAGE.get('origin') != 'random_initialization':
        raise RuntimeError('Refusing to resume a checkpoint without verified random-initialization lineage.')
    checkpoint_sha = hashlib.sha256(STATE_CHECKPOINT.read_bytes()).hexdigest()
    if STATE_RECORD.get('foundation_checkpoint_sha256') != checkpoint_sha:
        raise RuntimeError('Foundation checkpoint SHA-256 does not match state.json; refusing corrupted state.')
    tokenizer_bytes = STATE_TOKENIZER.read_bytes()
    external_sha = hashlib.sha256(tokenizer_bytes).hexdigest()
    recorded_tokenizer_sha = STATE_RECORD.get('tokenizer_sha256')
    if recorded_tokenizer_sha is not None and recorded_tokenizer_sha != external_sha:
        raise RuntimeError('Tokenizer SHA-256 does not match state.json; refusing corrupted state.')
    tokenizer_b64 = STATE_PAYLOAD.get('tokenizer_model_b64')
    if not isinstance(tokenizer_b64, str):
        raise RuntimeError('Foundation checkpoint does not embed its SentencePiece tokenizer.')
    embedded_tokenizer = base64.b64decode(tokenizer_b64, validate=True)
    if hashlib.sha256(embedded_tokenizer).hexdigest() != external_sha:
        raise RuntimeError('Persistent tokenizer does not match the tokenizer embedded in the foundation checkpoint.')
    CYCLE_INDEX = len(STATE_METADATA.get('runs') or [])
    if CYCLE_INDEX < 1:
        raise RuntimeError('Foundation checkpoint is missing its training-run lineage.')
    recorded_cycle = int(STATE_RECORD.get('training_cycle', -1))
    if recorded_cycle != CYCLE_INDEX:
        raise RuntimeError('Foundation checkpoint cycle does not match its state.json metadata.')
else:
    CYCLE_INDEX = 0

FINEWEB_GENERAL_SHUFFLE_SEED = 11337 + CYCLE_INDEX * 104729
FINEWEB_EDU_SHUFFLE_SEED = 1337 + CYCLE_INDEX * 104729
AYA_SHUFFLE_SEED = 3337 + CYCLE_INDEX * 104729
MIXTURE_SEED = 4337 + CYCLE_INDEX * 104729
TRAIN_SEED = 1337 + CYCLE_INDEX

print('Accelerator:', ACCELERATOR_NAME)
print('Precision:', PRECISION)
print('Training cycle:', CYCLE_INDEX + 1)
print('Resume foundation checkpoint:', HAS_FOUNDATION_STATE)
print('Training language: English only')
print('Sampling seeds:', {
    'fineweb_general': FINEWEB_GENERAL_SHUFFLE_SEED,
    'fineweb_edu': FINEWEB_EDU_SHUFFLE_SEED,
    'aya': AYA_SHUFFLE_SEED,
    'mixture': MIXTURE_SEED,
    'training': TRAIN_SEED,
})

### Data-use gate

The notebook records the source licence and review note in its manifest. The note is an audit entry, not legal advice or a substitute for reviewing the dataset card and upstream Common Crawl terms. If you have not accepted that research-use basis, stop before the download cell and choose another reviewed source.


In [ ]:
if (DATA_DIR / 'manifest.json').is_file():
    manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
    print('Using pinned English FineWeb-Edu data from the offline input bundle.')
else:
    cmd = [
        'artemus-prepare-hf', '--dataset', DATASET_ID,
        '--config', DATASET_CONFIG, '--split', DATASET_SPLIT,
        '--revision', DATASET_REVISION, '--license', DATASET_LICENSE,
        '--license-review-note', LICENSE_REVIEW_NOTE,
        '--output-dir', str(DATA_DIR),
        '--max-documents', str(MAX_DOCUMENTS),
        '--validation-percent', str(VALIDATION_PERCENT),
        '--min-chars', str(MIN_CHARS), '--max-chars', str(MAX_CHARS),
        '--required-language', 'en', '--min-language-score', str(MIN_LANGUAGE_SCORE),
        '--shuffle-seed', str(FINEWEB_EDU_SHUFFLE_SEED),
    ]
    subprocess.run(cmd, check=True)
    manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))

if manifest.get('language_policy', {}).get('required_language') != 'en':
    raise RuntimeError('FineWeb-Edu input was not prepared under the English-only language gate.')
if manifest.get('language_policy', {}).get('minimum_language_score', 0) < MIN_LANGUAGE_SCORE:
    raise RuntimeError('FineWeb-Edu input does not meet the required language-confidence threshold.')
print(json.dumps(manifest['counts'], indent=2))
print('Accepted English education documents:', manifest['accepted_documents'])
print('Sampling:', manifest['sampling'])

## 4. Add the broad English web stream

The general FineWeb sample supplies broad English web text; FineWeb-Edu supplies a smaller educationally filtered supplement. Both sources are prepared from their official training split only, require an explicit English language label and at least 0.70 language confidence, and retain immutable revisions and source provenance. The first foundation mixture is 80% general FineWeb and 20% FineWeb-Edu by document quota. Actual token shares are measured after fitting the Artemis tokenizer; document weights are not assumed to equal token weights.

Both sources declare ODC-By 1.0 and derive from Common Crawl. Attribution, underlying terms, residual privacy risks, and sample quality still require review. The pipeline is a bounded research pilot, not a full download of either corpus.

In [ ]:
GENERAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
if (GENERAL_DATA_DIR / 'manifest.json').is_file():
    general_manifest = json.loads((GENERAL_DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
    GENERAL_REVISION = general_manifest.get('resolved_revision') or general_manifest.get('dataset_revision')
    if not GENERAL_REVISION:
        raise RuntimeError('Prepared general FineWeb bundle has no immutable source revision.')
    print('Using pinned English general FineWeb data from the offline input bundle.')
else:
    from huggingface_hub import HfApi
    GENERAL_REVISION = HfApi().dataset_info(GENERAL_DATASET_ID).sha
    if not GENERAL_REVISION or len(GENERAL_REVISION) < 7:
        raise RuntimeError('Could not resolve an immutable general FineWeb revision.')
    general_cmd = [
        'artemus-prepare-hf', '--dataset', GENERAL_DATASET_ID,
        '--config', GENERAL_DATASET_CONFIG, '--split', 'train',
        '--revision', GENERAL_REVISION,
        '--license', GENERAL_DATASET_LICENSE,
        '--license-review-note', GENERAL_LICENSE_REVIEW_NOTE,
        '--output-dir', str(GENERAL_DATA_DIR),
        '--max-documents', str(GENERAL_MAX_DOCUMENTS),
        '--validation-percent', str(VALIDATION_PERCENT),
        '--min-chars', str(MIN_CHARS), '--max-chars', str(MAX_CHARS),
        '--required-language', 'en', '--min-language-score', str(MIN_LANGUAGE_SCORE),
        '--shuffle-seed', str(FINEWEB_GENERAL_SHUFFLE_SEED),
    ]
    subprocess.run(general_cmd, check=True)
    general_manifest = json.loads((GENERAL_DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
    general_manifest['resolved_revision'] = GENERAL_REVISION
    (GENERAL_DATA_DIR / 'manifest.json').write_text(
        json.dumps(general_manifest, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
    )

if general_manifest.get('language_policy', {}).get('required_language') != 'en':
    raise RuntimeError('General FineWeb input was not prepared under the English-only language gate.')
if general_manifest.get('language_policy', {}).get('minimum_language_score', 0) < MIN_LANGUAGE_SCORE:
    raise RuntimeError('General FineWeb input does not meet the required language-confidence threshold.')
print('General FineWeb revision:', GENERAL_REVISION)
print('Accepted English general documents:', general_manifest['accepted_documents'])
print('General FineWeb sampling:', general_manifest['sampling'])

### Build the English-only foundation mixture

The mixture is assembled from broad general English text and an educationally filtered English supplement. Deterministic document quotas, exact cross-source deduplication, and per-record provenance make the experiment reproducible. The 80/20 weighting is a pilot hypothesis; inspect actual per-source token shares and held-out loss before changing it.

In [ ]:
mixture_cmd = [
    'artemus-prepare-mixture', '--output-dir', str(MIX_DIR),
    '--train-documents', str(MIX_TRAIN_DOCUMENTS),
    '--validation-documents', str(MIX_VALIDATION_DOCUMENTS),
    '--seed', str(MIXTURE_SEED),
    '--source', 'fineweb-general-english', str(GENERAL_DATA_DIR / 'train.jsonl'), str(GENERAL_DATA_DIR / 'validation.jsonl'), str(FINEWEB_GENERAL_WEIGHT),
    '--source', 'fineweb-edu-english', str(DATA_DIR / 'train.jsonl'), str(DATA_DIR / 'validation.jsonl'), str(FINEWEB_EDU_WEIGHT),
]
subprocess.run(mixture_cmd, check=True)
mixture_manifest = json.loads((MIX_DIR / 'manifest.json').read_text(encoding='utf-8'))
print('Train mixture:', json.dumps(mixture_manifest['train'], indent=2, ensure_ascii=False))
print('Validation mixture:', json.dumps(mixture_manifest['validation'], indent=2, ensure_ascii=False))


## 4. Train a tokenizer on training text only

SentencePiece sees only the curated training documents, not validation. The model uses byte fallback and retains the dedicated chat control tokens for later supervised fine-tuning.


In [ ]:
from artemis.tokenizer import SentencePieceTokenizer
TOKENIZER_MODEL = TOKENIZER_PREFIX.with_suffix('.model')
if HAS_FOUNDATION_STATE:
    shutil.copy2(STATE_TOKENIZER, TOKENIZER_MODEL)
    tokenizer = SentencePieceTokenizer(model_file=TOKENIZER_MODEL)
else:
    tokenizer = SentencePieceTokenizer.train(
        MIX_DIR / 'train.txt', TOKENIZER_PREFIX, vocab_size=VOCAB_SIZE
    )
print('Tokenizer vocab:', tokenizer.vocab_size)
print('Tokenizer model:', TOKENIZER_PREFIX.with_suffix('.model'))


# Measure the real token mixture with the final tokenizer; document ratios are only a proxy.
from collections import Counter
mixture_token_counts = Counter()
for line in (MIX_DIR / 'train.jsonl').open('r', encoding='utf-8'):
    if line.strip():
        row = json.loads(line)
        mixture_token_counts[row['mixture']['source_id']] += len(tokenizer.encode(row['text']))
mixture_token_total = sum(mixture_token_counts.values())
mixture_token_shares = {
    key: value / mixture_token_total
    for key, value in mixture_token_counts.items()
}
mixture_manifest_path = MIX_DIR / 'manifest.json'
mixture_manifest = json.loads(mixture_manifest_path.read_text(encoding='utf-8'))
mixture_manifest['tokenizer_measurement'] = {
    'tokenizer': tokenizer.name,
    'vocab_size': tokenizer.vocab_size,
    'train_tokens_by_source': dict(mixture_token_counts),
    'train_token_share_by_source': mixture_token_shares,
}
mixture_manifest_path.write_text(json.dumps(mixture_manifest, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Actual token shares:', mixture_token_shares)


## 5. Continue foundation pretraining without resetting model weights

The first cycle starts from random initialization. Later cycles resume the previous **foundation** checkpoint, keep the exact same tokenizer and architecture, and train for another bounded segment on a newly shuffled sample. The separate instruction-tuned candidate is never fed back into foundation pretraining.

In [ ]:
# The state dataset contains only the foundation checkpoint, never the prior instruction-SFT candidate.
if HAS_FOUNDATION_STATE:
    shutil.copy2(STATE_CHECKPOINT, CHECKPOINT)
    previous_step = int(STATE_PAYLOAD['step'])
    target_step = previous_step + TRAIN_STEPS
    phase = 'continued_pretraining'
    resume_args = [
        '--resume', str(CHECKPOINT),
        '--allow-data-change',
        '--override-resume-learning-rate',
    ]
else:
    previous_step = 0
    target_step = TRAIN_STEPS
    phase = 'foundation_pretraining'
    resume_args = [
        '--tokenizer-model', str(TOKENIZER_PREFIX.with_suffix('.model')),
    ]

train_cmd = [
    'artemus-train', '--data', str(MIX_DIR / 'train.jsonl'),
    '--validation-data', str(MIX_DIR / 'validation.jsonl'),
    '--data-manifest', str(MIX_DIR / 'manifest.json'),
    '--document-format', 'jsonl', '--text-field', 'text',
    '--token-cache-dir', str(ARTIFACT_ROOT / 'token-cache'),
    '--tokenizer', 'sentencepiece',
    *resume_args,
    '--phase', phase, '--steps', str(target_step),
    '--batch-size', str(BATCH_SIZE), '--gradient-accumulation-steps', str(GRAD_ACCUM),
    '--precision', PRECISION, '--block-size', str(BLOCK_SIZE),
    '--layers', '4', '--heads', '4', '--embedding-size', '128', '--embedding-dim', '64',
    '--kv-heads', '2', '--use-rope', '--use-qk-norm', '--use-swiglu',
    '--learning-rate', str(BASE_LR), '--lr-schedule', 'cosine',
    '--min-learning-rate', '0.00003', '--warmup-steps', '100',
    '--eval-interval', str(FOUNDATION_EVAL_INTERVAL), '--checkpoint-interval', str(CHECKPOINT_INTERVAL),
    '--eval-iters', str(EVAL_ITERS), '--seed', str(TRAIN_SEED),
    '--device', 'auto', '--output', str(CHECKPOINT),
]
print(f'Foundation cycle {CYCLE_INDEX + 1}: {phase}, steps {previous_step} -> {target_step}')
subprocess.run(train_cmd, check=True)
assert CHECKPOINT.is_file() and CHECKPOINT.stat().st_size > 0
foundation_payload = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
assert int(foundation_payload['step']) == target_step
assert foundation_payload.get('response_only') is False
print('Foundation checkpoint bytes:', CHECKPOINT.stat().st_size)


## 7. Optional code-domain continuation with OSCR

OSCR scripts are **not** under one universal licence. Each repository manifest preserves the source licence and the evidence used to confirm it. Only select repository manifest paths whose exact SPDX licence appears in the tool's allowlist (MIT, Apache-2.0, BSD, ISC, CC0, etc.). Exclude NC/SA/unknown/compound licences unless a separate policy has approved them. OSCR recommends downloading only selected manifests and blocks, not crawling its site.

Add exact paths from the OSCR dataset's `manifests/.../*.json` tree below. Leave the list empty to skip this stage. The dataset revision is pinned in the script.


In [ ]:
OSCR_MANIFEST_PATHS = []  # e.g. ['manifests/ab/<repository-manifest>.json']
OSCR_DIR = ARTIFACT_ROOT / 'oscr'
OSCR_MIX_DIR = ARTIFACT_ROOT / 'code_replay_mixture'
if OSCR_MANIFEST_PATHS:
    oscr_cmd = ['artemus-prepare-oscr', '--output-dir', str(OSCR_DIR),
                '--dataset-revision', 'b0e3fadb276678880f7f3d764308fbebd3bcd044',
                '--max-files', '10000']
    for manifest_path in OSCR_MANIFEST_PATHS:
        oscr_cmd.extend(['--hub-manifest-path', manifest_path])
    subprocess.run(oscr_cmd, check=True)

    # Retain the general English foundation mixture during code adaptation.
    code_mix_cmd = [
        'artemus-prepare-mixture', '--output-dir', str(OSCR_MIX_DIR),
        '--train-documents', '10000', '--validation-documents', '500',
        '--seed', '1338',
        '--source', 'general-replay', str(MIX_DIR / 'train.jsonl'), str(MIX_DIR / 'validation.jsonl'), '0.85',
        '--source', 'python-code', str(OSCR_DIR / 'train.jsonl'), str(OSCR_DIR / 'validation.jsonl'), '0.15',
    ]
    subprocess.run(code_mix_cmd, check=True)
    parent_state = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
    code_target_step = int(parent_state['step']) + 3000
    CODE_CHECKPOINT = CHECKPOINT_ROOT / 'artemus-code-adapted.pt'
    subprocess.run([
        'artemus-train', '--data', str(OSCR_MIX_DIR / 'train.jsonl'),
        '--validation-data', str(OSCR_MIX_DIR / 'validation.jsonl'),
        '--data-manifest', str(OSCR_MIX_DIR / 'manifest.json'),
        '--document-format', 'jsonl', '--text-field', 'text',
        '--token-cache-dir', str(ARTIFACT_ROOT / 'oscr-token-cache'),
        '--tokenizer', 'sentencepiece',
        '--resume', str(CHECKPOINT), '--allow-data-change',
        '--phase', 'continued_pretraining', '--steps', str(code_target_step),
        '--batch-size', str(BATCH_SIZE), '--gradient-accumulation-steps', str(GRAD_ACCUM),
        '--precision', PRECISION, '--override-resume-learning-rate',
        '--learning-rate', '0.0001', '--lr-schedule', 'cosine', '--min-learning-rate', '0.00001',
        '--warmup-steps', '100', '--eval-interval', '250', '--checkpoint-interval', str(CHECKPOINT_INTERVAL), '--eval-iters', '8',
        '--device', 'auto', '--output', str(CODE_CHECKPOINT),
    ], check=True)
    CHECKPOINT = CODE_CHECKPOINT
else:
    print('OSCR stage skipped: no repository manifests selected.')


## 8. English-only instruction tuning with human-annotated Aya data

After foundation pretraining (and optional code adaptation), this stage performs response-only supervised fine-tuning on a bounded, pinned sample from Cohere's Aya dataset. The source carries explicit language labels; the preparation pipeline admits only records tagged with the English language code eng. Annotator identifiers and demographic fields are discarded. Human annotations still require factuality and regression checks.

In [ ]:
AYA_DIR.mkdir(parents=True, exist_ok=True)
if (AYA_DIR / 'manifest.json').is_file():
    aya_manifest = json.loads((AYA_DIR / 'manifest.json').read_text(encoding='utf-8'))
    aya_revision = aya_manifest.get('resolved_revision')
    if not aya_revision:
        raise RuntimeError('Prepared Aya bundle has no resolved source revision.')
    if aya_manifest.get('sampling', {}).get('allowed_language_codes') != ['eng']:
        raise RuntimeError('Prepared Aya data does not satisfy the English-only language gate.')
    print('Using pinned English-only Aya instruction data from the offline input bundle.')
else:
    from huggingface_hub import HfApi
    AYA_DATASET = 'CohereLabs/aya_dataset'
    aya_revision = HfApi().dataset_info(AYA_DATASET).sha
    if not aya_revision or len(aya_revision) < 7:
        raise RuntimeError('Could not resolve an immutable Aya dataset revision.')
    subprocess.run([
        'artemus-prepare-aya', '--revision', aya_revision,
        '--split', 'train', '--output-dir', str(AYA_DIR),
        '--max-examples', str(AYA_MAX_EXAMPLES), '--validation-percent', '2.0',
        '--shuffle-seed', str(AYA_SHUFFLE_SEED),
        '--language-codes', 'eng',
    ], check=True)
    aya_manifest = json.loads((AYA_DIR / 'manifest.json').read_text(encoding='utf-8'))
    aya_manifest['resolved_revision'] = aya_revision
    (AYA_DIR / 'manifest.json').write_text(
        json.dumps(aya_manifest, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
    )
print('Aya revision:', aya_revision)
print('Accepted examples:', aya_manifest['accepted_examples'])
print('English-only language counts:', aya_manifest['examples_by_language_code'])
print('Aya sampling:', aya_manifest['sampling'])

parent_state = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
aya_target_step = int(parent_state['step']) + AYA_SFT_STEPS
AYA_CHECKPOINT = CHECKPOINT_ROOT / 'artemus-aya-instruction-sft.pt'
subprocess.run([
    'artemus-train', '--data', str(AYA_DIR / 'train.jsonl'),
    '--validation-data', str(AYA_DIR / 'validation.jsonl'),
    '--data-manifest', str(AYA_DIR / 'manifest.json'),
    '--document-format', 'jsonl', '--text-field', 'text',
    '--token-cache-dir', str(ARTIFACT_ROOT / 'aya-token-cache'),
    '--tokenizer', 'sentencepiece', '--resume', str(CHECKPOINT),
    '--allow-data-change', '--allow-objective-change',
    '--phase', 'instruction_sft', '--response-only', '--steps', str(aya_target_step),
    '--batch-size', str(BATCH_SIZE), '--gradient-accumulation-steps', str(GRAD_ACCUM),
    '--precision', PRECISION, '--override-resume-learning-rate',
    '--learning-rate', '0.00005', '--lr-schedule', 'cosine', '--min-learning-rate', '0.000005',
    '--warmup-steps', '50', '--eval-interval', str(AYA_EVAL_INTERVAL),
    '--checkpoint-interval', str(CHECKPOINT_INTERVAL), '--eval-iters', str(EVAL_ITERS),
    '--device', 'auto', '--output', str(AYA_CHECKPOINT),
], check=True)
CHECKPOINT = AYA_CHECKPOINT
print('Current candidate checkpoint:', CHECKPOINT)


## 10. Task-specific supervised tuning: conversations, maths and Python

This stage reuses the repository's pinned OASST1, GSM8K and MBPP preparation code. GSM8K is split by stable hash buckets: bucket 0 is used for optimizer validation, bucket 1 is reserved for final diagnostics, and buckets 2-9 are used for training. MBPP IDs 511-555 are used for optimizer validation, 556-600 are diagnostic-only, and 601-974 are used for training. MBPP IDs 1-510 and the GSM8K official test file remain excluded.

The task candidate starts from this cycle's Aya-tuned checkpoint. It remains separate from the foundation checkpoint used by the next cycle. Evaluation examples are disjoint from all training and optimizer-validation examples.

In [ ]:
OASST_DIR = ARTIFACT_ROOT / 'oasst1'
CAPABILITY_DIR = ARTIFACT_ROOT / 'capability_sft'
RAW_DIR = ARTIFACT_ROOT / 'raw-capability-data'
RAW_DIR.mkdir(parents=True, exist_ok=True)

if (CAPABILITY_DIR / 'manifest.json').is_file():
    capability_manifest = json.loads((CAPABILITY_DIR / 'manifest.json').read_text(encoding='utf-8'))
    print('Using pinned OASST1/GSM8K/MBPP prepared data from the offline input bundle.')
else:
    from urllib.request import urlretrieve
    from artemis.prepare_oasst1 import SOURCE_URL as OASST1_URL, EXPECTED_SHA256 as OASST1_SHA256
    from artemis.prepare_capability_data import (
        GSM8K_URL, GSM8K_SHA256, MBPP_URL, MBPP_GIT_BLOB_SHA,
    )

    OASST_RAW = RAW_DIR / 'oasst1-ready.messages.jsonl.gz'
    GSM8K_RAW = RAW_DIR / 'gsm8k-train.jsonl'
    MBPP_RAW = RAW_DIR / 'mbpp-full.jsonl'
    for url, path in ((OASST1_URL, OASST_RAW), (GSM8K_URL, GSM8K_RAW), (MBPP_URL, MBPP_RAW)):
        if not path.is_file():
            urlretrieve(url, path)
        if path.stat().st_size == 0:
            raise RuntimeError(f'Training source is empty: {path.name}')

    import hashlib
    def sha256_file(path):
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(chunk)
        return digest.hexdigest()

    if sha256_file(OASST_RAW) != OASST1_SHA256:
        raise RuntimeError('OASST1 archive SHA-256 mismatch.')
    if sha256_file(GSM8K_RAW) != GSM8K_SHA256:
        raise RuntimeError('GSM8K official training file SHA-256 mismatch.')

    mbpp_bytes = MBPP_RAW.read_bytes()
    mbpp_header = b'blob ' + str(len(mbpp_bytes)).encode('ascii') + bytes([0])
    if hashlib.sha1(mbpp_header + mbpp_bytes).hexdigest() != MBPP_GIT_BLOB_SHA:
        raise RuntimeError('MBPP Git blob SHA mismatch.')

    OASST_MAX_EXAMPLES = min(5_000 + CYCLE_INDEX * 2_000, 30_000)
    subprocess.run([
        'python', '-m', 'artemis.prepare_oasst1',
        '--source', str(OASST_RAW),
        '--output-dir', str(OASST_DIR),
        '--max-examples', str(OASST_MAX_EXAMPLES),
    ], check=True)
    subprocess.run([
        'python', '-m', 'artemis.prepare_capability_data',
        '--oasst-train', str(OASST_DIR / 'train.txt'),
        '--oasst-validation', str(OASST_DIR / 'validation.txt'),
        '--gsm8k-train', str(GSM8K_RAW),
        '--mbpp-full', str(MBPP_RAW),
        '--output-dir', str(CAPABILITY_DIR),
    ], check=True)
    capability_manifest = json.loads((CAPABILITY_DIR / 'manifest.json').read_text(encoding='utf-8'))
print(json.dumps(capability_manifest, indent=2, ensure_ascii=False))

aya_parent = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
task_target_step = int(aya_parent['step']) + TASK_SFT_STEPS
TASK_CHECKPOINT = CHECKPOINT_ROOT / 'artemus-task-sft.pt'
task_train_cmd = [
    'artemus-train',
    '--data', str(CAPABILITY_DIR / 'train.txt'),
    '--validation-data', str(CAPABILITY_DIR / 'validation.txt'),
    '--data-manifest', str(CAPABILITY_DIR / 'manifest.json'),
    '--document-format', 'text',
    '--tokenizer', 'sentencepiece',
    '--resume', str(CHECKPOINT),
    '--allow-data-change',
    '--phase', 'task_sft',
    '--response-only',
    '--steps', str(task_target_step),
    '--batch-size', str(BATCH_SIZE),
    '--gradient-accumulation-steps', str(GRAD_ACCUM),
    '--precision', PRECISION,
    '--override-resume-learning-rate',
    '--learning-rate', '0.000025',
    '--lr-schedule', 'cosine',
    '--min-learning-rate', '0.000003',
    '--warmup-steps', '50',
    '--eval-interval', str(TASK_EVAL_INTERVAL),
    '--checkpoint-interval', str(CHECKPOINT_INTERVAL),
    '--eval-iters', str(EVAL_ITERS),
    '--seed', str(TRAIN_SEED + 17),
    '--device', 'auto',
    '--output', str(TASK_CHECKPOINT),
]
subprocess.run(task_train_cmd, check=True)
task_payload = torch.load(TASK_CHECKPOINT, map_location='cpu', weights_only=True)
assert int(task_payload['step']) == task_target_step
assert task_payload.get('response_only') is True
CHECKPOINT = TASK_CHECKPOINT
print('Task-SFT candidate:', CHECKPOINT, 'step:', task_payload['step'])


## 11. Generation health and held-out capability diagnostics

The dialogue check covers model loading, non-empty generation and short multi-turn context handling. A separate evaluator scores math exact-match and Python syntax diagnostics on the disjoint evaluation partition. Every cycle runs it against the foundation, Aya-tuned and task-SFT checkpoints using identical examples and decoding settings.

These are internal research diagnostics, not official benchmark scores. Python outputs are parsed but never executed; isolated functional tests and protected external test sets are still required before claiming coding capability.

In [ ]:
import hashlib

subprocess.run([
    'artemus-dialogue-eval', '--checkpoint', str(CHECKPOINT),
    '--scenarios', 'data/evaluation/dialogues.jsonl',
    '--output-dir', str(ARTIFACT_ROOT / 'dialogue-eval'),
    '--max-new-tokens', '64', '--top-k', '1', '--temperature', '1.0',
    '--seed', '1337', '--device', 'auto',
], check=True)
dialogue_summary = json.loads((ARTIFACT_ROOT / 'dialogue-eval' / 'summary.json').read_text())
print(json.dumps(dialogue_summary, indent=2))

CAPABILITY_EVAL_DIR = ARTIFACT_ROOT / 'capability-eval'
CAPABILITY_EVAL_STAGES = [
    ('foundation', CHECKPOINT_ROOT / 'artemus-foundation-fineweb-edu.pt'),
    ('aya', CHECKPOINT_ROOT / 'artemus-aya-instruction-sft.pt'),
    ('task-sft', CHECKPOINT_ROOT / 'artemus-task-sft.pt'),
]
capability_eval_summaries = {}
for stage_name, stage_checkpoint in CAPABILITY_EVAL_STAGES:
    if not stage_checkpoint.is_file():
        raise RuntimeError(f'Missing checkpoint for capability evaluation: {stage_checkpoint}')
    stage_output = CAPABILITY_EVAL_DIR / stage_name
    subprocess.run([
        'artemus-capability-eval',
        '--checkpoint', str(stage_checkpoint),
        '--data', str(CAPABILITY_DIR / 'evaluation.jsonl'),
        '--output-dir', str(stage_output),
        '--max-new-tokens', str(CAPABILITY_MAX_NEW_TOKENS),
        '--max-math-examples', str(CAPABILITY_MAX_MATH_EXAMPLES),
        '--max-code-examples', str(CAPABILITY_MAX_CODE_EXAMPLES),
        '--seed', '1337',
        '--device', 'auto',
    ], check=True)
    stage_summary = json.loads((stage_output / 'summary.json').read_text(encoding='utf-8'))
    if stage_summary['checkpoint_sha256'] != hashlib.sha256(stage_checkpoint.read_bytes()).hexdigest():
        raise RuntimeError(f'Capability evaluation checkpoint hash mismatch: {stage_name}')
    capability_eval_summaries[stage_name] = stage_summary

capability_eval_summary = capability_eval_summaries['task-sft']
evaluation_data_hashes = {
    value['evaluation_data_sha256'] for value in capability_eval_summaries.values()
}
if len(evaluation_data_hashes) != 1:
    raise RuntimeError('Stage evaluations did not use the exact same held-out input file.')
capability_eval_comparison = {
    'schema_version': 1,
    'evaluation_data_sha256': next(iter(evaluation_data_hashes)),
    'stages': {
        name: {
            'checkpoint_sha256': value['checkpoint_sha256'],
            'checkpoint_step': value['checkpoint_step'],
            'math': value['math'],
            'python': value['python'],
            'skipped_context': value['skipped_context'],
        }
        for name, value in capability_eval_summaries.items()
    },
    'interpretation': (
        'Compare diagnostics only across checkpoints using the same examples and decoding settings. '
        'These are internal held-out metrics, not official benchmark scores.'
    ),
}
comparison_path = ARTIFACT_ROOT / 'capability-evaluation-comparison.json'
comparison_path.write_text(
    json.dumps(capability_eval_comparison, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)
print(json.dumps(capability_eval_comparison, indent=2, ensure_ascii=False))


## 12. Human teaching and supervised fine-tuning

You can talk to the model in a local terminal using `artemus-teach --checkpoint <checkpoint>`. It asks you to correct, approve, or skip each answer. Corrections are saved as human-reviewed candidates; model-generated answers are not included unless you explicitly enable the approval flag. Review the candidate JSONL before using it.

To use a reviewed candidate file as SFT data, upload it as a Kaggle input and run a separate stage with `--response-only --allow-data-change --allow-objective-change`. Do not combine arbitrary unreviewed chat logs with the training corpus.


## 13. Export checkpoint, all data-stage provenance, and run lineage

The export bundle contains the final candidate, any intermediate foundation/code checkpoints, the tokenizer, every available source and mixture manifest/provenance file, dialogue evaluation traces, the exact repository commit, the checkpoint's training lineage, and SHA-256 hashes for every exported artifact. Raw training corpora are not copied into the bundle. Keep the ZIP with the experiment record; do not commit model weights or datasets to Git.

In [ ]:
# Export an auditable, self-contained run bundle without raw corpora.
import hashlib
from datetime import datetime, timezone

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPORT_DIR = WORK_ROOT / f"artemus-run-{RUN_STAMP}"
EXPORT_DIR.mkdir(parents=True, exist_ok=False)

def sha256_path(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Preserve only stage checkpoints expected from this run; never sweep up arbitrary
# stale .pt files left in a reused notebook workspace.
checkpoint_paths = [
    CHECKPOINT_ROOT / "artemus-foundation-fineweb-edu.pt",
    CHECKPOINT_ROOT / "artemus-aya-instruction-sft.pt",
    CHECKPOINT_ROOT / "artemus-task-sft.pt",
]
if OSCR_MANIFEST_PATHS:
    checkpoint_paths.insert(1, CHECKPOINT_ROOT / "artemus-code-adapted.pt")
if CHECKPOINT.is_file() and CHECKPOINT not in checkpoint_paths:
    checkpoint_paths.append(CHECKPOINT)
for checkpoint_path in checkpoint_paths:
    if checkpoint_path.is_file():
        shutil.copy2(checkpoint_path, EXPORT_DIR / checkpoint_path.name)

# Keep each stage's manifest and provenance beside the weights they describe.
stage_sources = [
    ("fineweb-general-english", GENERAL_DATA_DIR),
    ("fineweb-edu-english", DATA_DIR),
    ("foundation-mixture", MIX_DIR),
    ("aya-instruction-sft", AYA_DIR),
    ("oasst1", OASST_DIR),
    ("capability-sft", CAPABILITY_DIR),
]
if OSCR_MANIFEST_PATHS:
    stage_sources.append(("oscr-code", OSCR_DIR))

exported_source_files = []
for label, source_dir in stage_sources:
    if not source_dir.is_dir():
        raise RuntimeError(f"Expected training stage directory is missing: {source_dir}")
    for filename in ("manifest.json", "provenance.jsonl"):
        source_path = source_dir / filename
        if source_path.is_file():
            target_path = EXPORT_DIR / f"{label}-{filename}"
            shutil.copy2(source_path, target_path)
            exported_source_files.append(target_path.name)
    if not (EXPORT_DIR / f"{label}-manifest.json").is_file():
        raise RuntimeError(f"Missing exported source manifest for {label}")

# The final tokenizer and evaluation trace are part of the model's reproducibility record.
tokenizer_path = TOKENIZER_PREFIX.with_suffix(".model")
if not tokenizer_path.is_file():
    raise RuntimeError(f"Tokenizer model is missing: {tokenizer_path}")
shutil.copy2(tokenizer_path, EXPORT_DIR / "tokenizer.model")

dialogue_dir = ARTIFACT_ROOT / "dialogue-eval"
for filename in ("summary.json", "dialogues.jsonl"):
    source_path = dialogue_dir / filename
    if source_path.is_file():
        shutil.copy2(source_path, EXPORT_DIR / f"dialogue-{filename}")

capability_eval_dir = ARTIFACT_ROOT / "capability-eval"
for stage_name in ("foundation", "aya", "task-sft"):
    stage_dir = capability_eval_dir / stage_name
    for source_name, export_name in (
        ("summary.json", f"capability-evaluation-{stage_name}-summary.json"),
        ("results.jsonl", f"capability-evaluation-{stage_name}-results.jsonl"),
    ):
        source_path = stage_dir / source_name
        if not source_path.is_file():
            raise RuntimeError(f"Capability evaluation output is missing: {source_path}")
        shutil.copy2(source_path, EXPORT_DIR / export_name)
    if stage_name == "task-sft":
        shutil.copy2(stage_dir / "summary.json", EXPORT_DIR / "capability-evaluation-summary.json")
        shutil.copy2(stage_dir / "results.jsonl", EXPORT_DIR / "capability-evaluation-results.jsonl")

comparison_path = ARTIFACT_ROOT / "capability-evaluation-comparison.json"
if not comparison_path.is_file():
    raise RuntimeError("Stage-by-stage capability evaluation comparison is missing.")
shutil.copy2(comparison_path, EXPORT_DIR / "capability-evaluation-comparison.json")

evaluation_input = CAPABILITY_DIR / "evaluation.jsonl"
if not evaluation_input.is_file():
    raise RuntimeError(f"Structured held-out evaluation input is missing: {evaluation_input}")
shutil.copy2(evaluation_input, EXPORT_DIR / "capability-evaluation-input.jsonl")

# Preserve the resolved code revision and the checkpoint's complete training lineage.
checkpoint_payload = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
run_metadata = checkpoint_payload.get("run_metadata") or {}

# Persistent state is the foundation checkpoint, not the final task-SFT candidate.
foundation_checkpoint_export = EXPORT_DIR / "artemus-foundation-fineweb-edu.pt"
if not foundation_checkpoint_export.is_file():
    raise RuntimeError("The foundation checkpoint was not included in the export bundle.")
foundation_checkpoint_payload = torch.load(
    foundation_checkpoint_export, map_location="cpu", weights_only=True
)
foundation_run_metadata = foundation_checkpoint_payload.get("run_metadata") or {}
foundation_global_step = int(foundation_checkpoint_payload.get("step", 0))
foundation_training_run_count = len(foundation_run_metadata.get("runs") or [])
if foundation_global_step < 1 or foundation_training_run_count != CYCLE_INDEX + 1:
    raise RuntimeError("Foundation checkpoint progress does not match this training cycle.")
foundation_checkpoint_sha256 = sha256_path(foundation_checkpoint_export)

from artemis.config import ModelConfig
from artemis.model import ArtemisLM

size_model = ArtemisLM(ModelConfig.from_dict(checkpoint_payload["model_config"]))
model_parameters = sum(parameter.numel() for parameter in size_model.parameters())
model_weight_bytes = sum(
    parameter.numel() * parameter.element_size() for parameter in size_model.parameters()
)
model_weight_dtype = str(next(size_model.parameters()).dtype)
del size_model
model_size_report = {
    "model_parameters": model_parameters,
    "model_weight_bytes": model_weight_bytes,
    "model_weight_mib": round(model_weight_bytes / (1024 ** 2), 4),
    "model_parameter_dtype": model_weight_dtype,
    "final_checkpoint_bytes": CHECKPOINT.stat().st_size,
    "foundation_checkpoint_bytes": foundation_checkpoint_export.stat().st_size,
    "accelerator": ACCELERATOR_NAME,
    "cpu_fallback": CPU_FALLBACK,
    "precision_mode": PRECISION,
    "training_steps": {
        "foundation": TRAIN_STEPS,
        "aya_instruction_sft": AYA_SFT_STEPS,
        "task_sft": TASK_SFT_STEPS,
    },
    "training_profile": {
        "block_size": BLOCK_SIZE,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUM,
        "max_general_english_documents": GENERAL_MAX_DOCUMENTS,
        "max_education_english_documents": MAX_DOCUMENTS,
        "max_aya_english_examples": AYA_MAX_EXAMPLES,
        "language_scope": "English only",
    },
}
(EXPORT_DIR / "model-size.json").write_text(
    json.dumps(model_size_report, indent=2) + "\n", encoding="utf-8"
)

run_summary = {
    "schema_version": 1,
    "repository": "https://github.com/FAHTHIZ/Artemus",
    "repository_commit": REPO_COMMIT,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "final_checkpoint": CHECKPOINT.name,
    "checkpoint_sha256": sha256_path(CHECKPOINT),
    "global_step": int(checkpoint_payload.get("step", 0)),
    "final_candidate_global_step": int(checkpoint_payload.get("step", 0)),
    "foundation_checkpoint_sha256": foundation_checkpoint_sha256,
    "foundation_global_step": foundation_global_step,
    "foundation_training_run_count": foundation_training_run_count,
    "training_cycle": CYCLE_INDEX + 1,
    "cycle_seeds": {
        "fineweb_general": FINEWEB_GENERAL_SHUFFLE_SEED,
        "fineweb_edu": FINEWEB_EDU_SHUFFLE_SEED,
        "aya": AYA_SHUFFLE_SEED,
        "mixture": MIXTURE_SEED,
        "training": TRAIN_SEED,
    },
    "model_config": checkpoint_payload.get("model_config"),
    "model_parameters": model_parameters,
    "model_weight_bytes": model_weight_bytes,
    "accelerator": ACCELERATOR_NAME,
    "cpu_fallback": CPU_FALLBACK,
    "run_metadata": run_metadata,
    "source_artifacts": sorted(exported_source_files),
}
(EXPORT_DIR / "run-metadata.json").write_text(
    json.dumps(run_summary, indent=2, ensure_ascii=False, default=str) + "\n",
    encoding="utf-8",
)

# Hash every exported file. The manifest intentionally does not hash itself.
artifact_records = []
for path in sorted(p for p in EXPORT_DIR.rglob("*") if p.is_file()):
    artifact_records.append({
        "path": path.relative_to(EXPORT_DIR).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_path(path),
    })
export_manifest = {
    "schema_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repository_commit": REPO_COMMIT,
    "final_checkpoint": CHECKPOINT.name,
    "artifacts": artifact_records,
}
(EXPORT_DIR / "export-manifest.json").write_text(
    json.dumps(export_manifest, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

EXPORT_ZIP = shutil.make_archive(str(WORK_ROOT / f"artemus-run-{RUN_STAMP}"), "zip", EXPORT_DIR)
print("Export directory:", EXPORT_DIR)
print("Export archive:", EXPORT_ZIP)
print("Exported files:", [p.name for p in sorted(EXPORT_DIR.iterdir())])
print("Data-stage provenance files:", exported_source_files)


## 14. Next steps

1. Compare the foundation, Aya-tuned, and task-SFT checkpoints on identical held-out evaluation inputs and decoding budgets.
2. Evaluate an English-only alternative pretraining source such as DCLM or Dolma as a separate matched-budget experiment; deduplicate against FineWeb before mixing.
3. Add English mathematical web text such as OpenWebMath only as a separately measured domain-adaptation stage, never as a replacement for verified math tasks.
4. Add human-preference data only after a preference-training objective and independent evaluation are implemented. Anthropic HH-RLHF is preference/reward-model data, not dialogue SFT.
5. Consider teacher-generated training candidates from openly licensed model outputs only with source/version provenance, independent verification, and human review.
6. Never train indefinitely on one sample: advance to a new pinned English data release or stop when held-out validation plateaus or regresses.